# CM_metrics_text

Computes all the quantitative metrics needed to fill the placeholders in the
*Tension-Dependent Shifts in Node Influence Across Fibroblast Phenotypes* section,
and prints the filled-in LaTeX text at the bottom (ready to paste into Overleaf).

**Inputs**: canonical Set B scalars + calibrated 5-family weights.
**Outputs**: four KD matrices (one per tension level), a CSV of every (phenotype, KD, context, ΔActivity) cell, and the cohesive LaTeX paragraph block.

In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import ode

import NetfluxODE_5
import NetfluxODE_5_sensitivity
import NetfluxODE_5_params
from CM_model_DOF_clustering_5 import build_weight_vector, family_id_vector, wlen

[speciesNames, tau, ymax, y0_init, _, _, _] = NetfluxODE_5_params.loadParams()

# Canonical Set B scalars (paper baseline)
EC50_scalar = 0.587798
n_scalar    = 1.395697
b_scalar    = 0.283201
p_scalar    = 0.653814

calibrated_weights = np.array([1.01915194, 1.09999999, 0.94966909, 1.10, 0.62873844])
w_vec = build_weight_vector(calibrated_weights, family_id_vector)
EC50_v = EC50_scalar * np.ones(wlen)
n_v    = n_scalar    * np.ones(wlen)

input_list = ['mechanical','infusionTGFB','AngII','IL6','FGF','PDGF','ET1','TNFa','IL1']
input_indices = {nm: speciesNames.index(nm) for nm in input_list}

# Fresh y0 with all inputs at b_scalar
y0 = 0.1 * np.ones(len(y0_init))
for nm in input_list:
    y0[input_indices[nm]] = b_scalar

mech_idx = speciesNames.index('mechanical')
M = len(speciesNames)
print('species:', M, ' weights:', wlen)

species: 96  weights: 163


In [2]:
def simulate_ss(y0_vec):
    r = ode(NetfluxODE_5.ODEfunc).set_integrator(
        'vode', method='adams', order=10, rtol=0, atol=1e-6, with_jacobian=False)
    r.set_initial_value(y0_vec, 0.0).set_f_params(tau, ymax, w_vec, n_v, EC50_v)
    tf, dt = 60.0, 60.0/6000.0
    while r.successful() and r.t <= tf:
        r.integrate(r.t + dt)
    return r.y.copy()

def kd_matrix(y0_vec):
    """Return DataFrame: rows=measured species, cols=KD species, values=Δ activity (KD-control)."""
    baseline_ss = simulate_ss(y0_vec)
    SA = np.empty((M, M), dtype=float)
    for i in range(M):
        S = np.eye(M); S[i, i] = 0.0
        y0_kd = S @ y0_vec
        r = ode(NetfluxODE_5_sensitivity.ODEfunc).set_integrator(
            'vode', method='adams', order=10, rtol=0, atol=1e-6, with_jacobian=False)
        r.set_initial_value(y0_kd, 0.0).set_f_params(tau, ymax, w_vec, n_v, EC50_v, S)
        tf, dt = 60.0, 60.0/6000.0
        while r.successful() and r.t <= tf:
            r.integrate(r.t + dt)
        SA[i, :] = r.y - baseline_ss
    return pd.DataFrame(SA.T, index=speciesNames, columns=speciesNames)  # rows=measured, cols=KD

tension_levels = {
    'low':      0.01,
    'baseline': float(b_scalar),
    'high':     float(b_scalar + p_scalar),
    'max':      1.00,
}

context_M = {}
for name, tens in tension_levels.items():
    y0_ctx = y0.copy()
    y0_ctx[mech_idx] = float(tens)
    print(f'computing KD matrix at {name} (mechanical={tens:.3f})...')
    context_M[name] = kd_matrix(y0_ctx)
print('done.')

computing KD matrix at low (mechanical=0.010)...
computing KD matrix at baseline (mechanical=0.283)...
computing KD matrix at high (mechanical=0.937)...
computing KD matrix at max (mechanical=1.000)...
done.


In [3]:
# Save full matrices to disk for auditing
for name, df in context_M.items():
    df.to_csv(f'KD_matrix_{name}.csv')
print('saved KD_matrix_{low,baseline,high,max}.csv')

saved KD_matrix_{low,baseline,high,max}.csv


In [4]:
# ------------------- query helpers -------------------
phenotypes = ['aSMA','proMMP1','proMMP2','proMMP9','MMP1','MMP2','MMP9',
              'CImRNA','CI','fibronectin','proliferation']
THRESH = 0.25

def delta(kd_node, phen, ctx):
    """Return ΔActivity (KD - control) for one cell."""
    return float(context_M[ctx].loc[phen, kd_node])

def deltas(kd_nodes, phen, ctx):
    """Return list of ΔActivity values for several KDs on one phenotype."""
    return [delta(k, phen, ctx) for k in kd_nodes if k in context_M[ctx].columns]

def range_str(vals, fmt='{:+.2f}'):
    if not vals: return 'NA'
    if len(vals) == 1: return fmt.format(vals[0])
    return f'{fmt.format(min(vals))} to {fmt.format(max(vals))}'

def n_above_threshold_per_phen(ctx, phen, thr=THRESH):
    row = context_M[ctx].loc[phen].drop(labels=[phen], errors='ignore')
    return int((row.abs() >= thr).sum())

def n_brakes_per_phen(ctx, phen, thr=THRESH):
    row = context_M[ctx].loc[phen].drop(labels=[phen], errors='ignore')
    return int(((row >= thr)).sum())

def n_boosters_per_phen(ctx, phen, thr=THRESH):
    row = context_M[ctx].loc[phen].drop(labels=[phen], errors='ignore')
    return int(((row <= -thr)).sum())

def n_phenotypes_with_brakes(ctx, thr=THRESH):
    return sum(1 for p in phenotypes if n_brakes_per_phen(ctx, p, thr) > 0)

def n_phenotype_top10s_with_kd(kd_node, ctx):
    """How many of the 11 phenotype top-10 lists contain this KD at this context."""
    count = 0
    for p in phenotypes:
        row = context_M[ctx].loc[p].drop(labels=[p], errors='ignore').abs()
        top10 = set(row.sort_values(ascending=False).head(10).index)
        if kd_node in top10:
            count += 1
    return count

# Sanity print
print('Δ(LATS12, CImRNA, baseline) =', delta('LATS12','CImRNA','baseline'))
print('Δ(LATS12, CImRNA, high)     =', delta('LATS12','CImRNA','high'))
print('# brakes on aSMA at baseline:', n_brakes_per_phen('baseline','aSMA'))
print('# brakes on aSMA at high    :', n_brakes_per_phen('high','aSMA'))

Δ(LATS12, CImRNA, baseline) = 0.9271163207291612
Δ(LATS12, CImRNA, high)     = 2.403632848313464e-13
# brakes on aSMA at baseline: 0
# brakes on aSMA at high    : 0


In [5]:
# ------------------- node groups used in the text -------------------
G_distributed_boosters = ['infusionTGFB','TGFB','ET1','AngII','mechanical','ROS','AP1','NOX','ERK']
G_il6_axis             = ['IL6','gp130','STAT']
G_lats                 = ['LATS12']
G_mechanotransduction  = ['B1int','Rho','ROCK','Factin','SRF']        # integrin-Rho-ROCK-F-actin-SRF
G_tgfb_brake           = ['TGFB1R','smad3']
G_proMMP1_amplified    = ['TGFB','ROS','NOX','AP1','mechanical','NFKB']
G_MMP1_baseline_boost  = ['p38']
G_MMP1_high_boost      = ['TGFB','TGFB1R','infusionTGFB','mechanical']
G_MMP2_baseline_boost  = ['IL6','gp130','STAT','NFKB','p38']
G_MMP2_high_boost      = ['ROS','NOX','TGFB']
G_TIMPs                = ['TIMP1','TIMP2']
G_YAP_TEAD             = ['YAP','TEAD']
G_CI_brakes            = ['MMP1','MMP2','MMP14']
G_CI_brakes_upstream   = ['TGFB1R','ROS','AP1','NOX']
G_CI_weak_boosters     = ['CImRNA','B1int','Rho','ROCK','TIMP1']

groups = {
 'distributed_boosters' : G_distributed_boosters,
 'IL6-gp130-STAT'       : G_il6_axis,
 'LATS1/2'              : G_lats,
 'mechanotransduction'  : G_mechanotransduction,
 'TGFb-brake'           : G_tgfb_brake,
}
for g, members in groups.items():
    miss = [m for m in members if m not in speciesNames]
    if miss: print(f'WARNING: {g} missing: {miss}')

In [6]:
# ------------------- gather every metric the text needs -------------------
Q = {}

# Aggregate counts
Q['n_phen_with_brakes_baseline'] = n_phenotypes_with_brakes('baseline')
Q['n_phen_with_brakes_high']     = n_phenotypes_with_brakes('high')
Q['n_phen_with_brakes_max']      = n_phenotypes_with_brakes('max')

# Per-phenotype regulator counts (above 0.25)
for ctx in ['baseline','high','max']:
    for p in phenotypes:
        Q[f'n_above_{p}_{ctx}']    = n_above_threshold_per_phen(ctx, p)
        Q[f'n_boost_{p}_{ctx}']    = n_boosters_per_phen(ctx, p)
        Q[f'n_brake_{p}_{ctx}']    = n_brakes_per_phen(ctx, p)

# αSMA: max |Δ| of any regulator at baseline (to support 'least sensitive' claim)
for ctx in ['baseline','high','max']:
    row = context_M[ctx].loc['aSMA'].drop('aSMA').abs()
    Q[f'aSMA_max_abs_{ctx}'] = float(row.max())

# Distributed boosters at baseline → broad MMP/fibronectin/proliferation
_ds_targets = ['MMP1','MMP2','proMMP1','proMMP2','proMMP9','fibronectin','proliferation']
_ds_vals_b = []
_ds_vals_h = []
for p in _ds_targets:
    for k in G_distributed_boosters:
        _ds_vals_b.append(delta(k, p, 'baseline'))
        _ds_vals_h.append(delta(k, p, 'high'))
Q['distrib_boost_range_baseline'] = range_str([v for v in _ds_vals_b if v < 0])
Q['distrib_boost_range_high']     = range_str([v for v in _ds_vals_h if v < 0])

# IL6-gp130-STAT axis on (pro)MMP2/9 at baseline
_il6_b = [delta(k, p, 'baseline') for k in G_il6_axis for p in ['proMMP2','proMMP9','MMP2','MMP9']]
_il6_h = [delta(k, p, 'high')     for k in G_il6_axis for p in ['proMMP2','proMMP9','MMP2','MMP9']]
Q['IL6_axis_proMMP29_baseline']   = range_str(_il6_b)
Q['IL6_axis_proMMP29_high']       = range_str(_il6_h)

# IL6-gp130-STAT on proMMP1 at baseline (weak)
Q['IL6_axis_proMMP1_baseline']    = range_str([delta(k,'proMMP1','baseline') for k in G_il6_axis])

# LATS1/2 on collagen and proliferation at baseline
Q['LATS_CI_baseline']         = delta('LATS12','CI','baseline')
Q['LATS_CImRNA_baseline']     = delta('LATS12','CImRNA','baseline')
Q['LATS_proliferation_baseline'] = delta('LATS12','proliferation','baseline')
Q['LATS_CI_high']             = delta('LATS12','CI','high')
Q['LATS_CImRNA_high']         = delta('LATS12','CImRNA','high')
Q['LATS_proliferation_high']  = delta('LATS12','proliferation','high')

# αSMA at high tension: mechanotransduction boosters and TGFb brakes
Q['aSMA_mechano_high']  = range_str([delta(k,'aSMA','high') for k in G_mechanotransduction])
Q['aSMA_tgfb_brake_high']  = range_str([delta(k,'aSMA','high') for k in G_tgfb_brake])

# proMMP1 amplified boosters at baseline vs high
Q['proMMP1_amp_baseline']  = range_str([delta(k,'proMMP1','baseline') for k in G_proMMP1_amplified])
Q['proMMP1_amp_high']      = range_str([delta(k,'proMMP1','high') for k in G_proMMP1_amplified])
Q['proMMP1_smad7_baseline']= delta('smad7','proMMP1','baseline') if 'smad7' in speciesNames else None
Q['proMMP1_smad7_high']    = delta('smad7','proMMP1','high') if 'smad7' in speciesNames else None
Q['proMMP1_smad3_baseline']= delta('smad3','proMMP1','baseline')
Q['proMMP1_smad3_high']    = delta('smad3','proMMP1','high')

# MMP1 — p38 baseline vs high
Q['MMP1_p38_baseline']     = delta('p38','MMP1','baseline')
Q['MMP1_p38_high']         = delta('p38','MMP1','high')
Q['MMP1_TGFb_high']        = range_str([delta(k,'MMP1','high') for k in G_MMP1_high_boost])

# MMP2
Q['MMP2_baseline_boost']   = range_str([delta(k,'MMP2','baseline') for k in G_MMP2_baseline_boost])
Q['MMP2_high_boost_old']   = range_str([delta(k,'MMP2','high')     for k in G_MMP2_baseline_boost])
Q['MMP2_high_boost_new']   = range_str([delta(k,'MMP2','high')     for k in G_MMP2_high_boost])
Q['MMP2_TIMP1_baseline']   = delta('TIMP1','MMP2','baseline')
Q['MMP2_TIMP1_high']       = delta('TIMP1','MMP2','high')

# MMP9
Q['MMP9_il6_baseline']     = range_str([delta(k,'MMP9','baseline') for k in G_il6_axis])
Q['MMP9_NFKB_baseline']    = delta('NFKB','MMP9','baseline')
Q['MMP9_NFKB_high']        = delta('NFKB','MMP9','high')
Q['MMP9_TIMPs_baseline']   = range_str([delta(k,'MMP9','baseline') for k in G_TIMPs])
Q['MMP9_TIMPs_high']       = range_str([delta(k,'MMP9','high') for k in G_TIMPs])

# CImRNA
Q['CImRNA_LATS_baseline']  = delta('LATS12','CImRNA','baseline')
Q['CImRNA_LATS_high']      = delta('LATS12','CImRNA','high')
Q['CImRNA_mechano_high']   = range_str([delta(k,'CImRNA','high') for k in ['B1int','Rho','ROCK']])
Q['CImRNA_YAP_TEAD_high']  = range_str([delta(k,'CImRNA','high') for k in G_YAP_TEAD])

# CI (protein)
Q['CI_MMPs_high']          = range_str([delta(k,'CI','high') for k in G_CI_brakes])
Q['CI_upstream_high']      = range_str([delta(k,'CI','high') for k in G_CI_brakes_upstream])
Q['CI_weak_boost_high']    = range_str([delta(k,'CI','high') for k in G_CI_weak_boosters])
Q['CI_LATS_baseline']      = delta('LATS12','CI','baseline')
Q['CI_LATS_high']          = delta('LATS12','CI','high')
Q['CI_YAP_TEAD_high']      = range_str([delta(k,'CI','high') for k in G_YAP_TEAD])

# Fibronectin / proliferation: counts of boosters above threshold + key cells at high
Q['fibro_n_boost_baseline'] = n_boosters_per_phen('baseline','fibronectin')
Q['fibro_n_boost_high']     = n_boosters_per_phen('high','fibronectin')
Q['fibro_mech_high']        = delta('mechanical','fibronectin','high')
Q['fibro_NFKB_high']        = delta('NFKB','fibronectin','high')
Q['prol_n_boost_baseline']  = n_boosters_per_phen('baseline','proliferation')
Q['prol_n_boost_high']      = n_boosters_per_phen('high','proliferation')
Q['prol_mech_high']         = delta('mechanical','proliferation','high')

# Integrin (B1int) and mechanical input as emergent regulators
Q['B1int_top10_baseline'] = n_phenotype_top10s_with_kd('B1int','baseline')
Q['B1int_top10_high']     = n_phenotype_top10s_with_kd('B1int','high')
Q['B1int_aSMA_high']      = delta('B1int','aSMA','high')
Q['B1int_proMMP1_high']   = delta('B1int','proMMP1','high')
Q['mech_top10_baseline']  = n_phenotype_top10s_with_kd('mechanical','baseline')
Q['mech_top10_high']      = n_phenotype_top10s_with_kd('mechanical','high')
Q['mech_top10_max']       = n_phenotype_top10s_with_kd('mechanical','max')

# max-tension comparison: amplification vs new emergence
for p in phenotypes:
    h = context_M['high'].loc[p].drop(p).abs()
    m = context_M['max'].loc[p].drop(p).abs()
    Q[f'max_minus_high_mean_{p}'] = float((m - h).mean())

# Save quick metrics CSV for audit
pd.Series(Q).to_csv('CM_metrics_text_values.csv', header=['value'])
print('saved CM_metrics_text_values.csv')
print('total metrics:', len(Q))

saved CM_metrics_text_values.csv
total metrics: 172


In [7]:
# ------------------- format helpers -------------------
def fmt(v, k=2):
    if v is None: return 'NA'
    if isinstance(v, (int, np.integer)): return f'{int(v)}'
    return f'{float(v):+.{k}f}'

def G(name): return Q[name]
def F(name, k=2):
    v = Q[name]
    if isinstance(v, str): return v   # already a range string
    return fmt(v, k)

In [8]:
# ------------------- assemble the LaTeX text -------------------
TEXT = rf"""
Fibroblast signaling and ECM remodeling pathways have been mapped extensively, yet the relative influence of individual regulators within these pathways remains poorly understood, especially under varying mechanical tension. To address this gap, we quantified how each upstream species and input perturbation shapes phenotypic outputs across four tension levels, defined as low ($y_{{\text{{mechanical}}}} = 0.01$), baseline ($y_{{\text{{mechanical}}}} = b$), high ($y_{{\text{{mechanical}}}} = b + p$), and maximum ($y_{{\text{{mechanical}}}} = 1$). For each tension level we identify ``brakes'' (nodes whose knockout increases phenotypic activity, $\Delta X_{{iKO}} > 0$) and ``boosters'' (nodes whose knockout decreases phenotypic activity, $\Delta X_{{iKO}} < 0$), and quantify their effect on eleven phenotypic nodes ($\alpha$SMA, proMMP1, proMMP2, proMMP9, MMP1, MMP2, MMP9, CImRNA, CI, fibronectin, and proliferation; Fig.~\ref{{case3}}). On each panel of Fig.~\ref{{case3}}A--D, the columns represent the union of the top-10 most influential knockdowns identified for each phenotype within each tension context (eleven phenotypes $\times$ four tensions $=$ 44 lists). Columns are ordered left-to-right by how frequently each node appears across these lists, with ties broken by the average magnitude of its knockdown effect across phenotypes and contexts. As a result, leftmost columns correspond to broadly influential signaling hubs, while rightmost columns identify nodes whose influence is more phenotype- or tension-specific.

At low and baseline tension levels, regulatory influence is mostly activation-driven (Fig.~\ref{{case3}}A--B and Fig.~S2A--B). Nine of the eleven phenotypes are controlled by distributed booster modules, with only collagen-related nodes (CI, CImRNA) and proliferation showing a single consistent brake (LATS1/2, with $\Delta\text{{Activity}} = {F('LATS_CI_baseline')}$ on CI, ${F('LATS_CImRNA_baseline')}$ on CImRNA, and ${F('LATS_proliferation_baseline')}$ on proliferation). At baseline, {G('n_phen_with_brakes_baseline')} of the 11 phenotypes show any brake above the influence threshold $|\Delta\text{{Activity}}| \geq 0.25$, while the remaining phenotypes are entirely booster-controlled.

Several signaling modules act as broad boosters across multiple phenotypes. Input nodes (TGF$\beta$, ET1, AngII, mechanical) and intermediate nodes (ROS, AP1, NOX, ERK) collectively produce a subtle but distributed activation of MMP1/2, proMMP1/2/9, fibronectin, and proliferation, with $\Delta\text{{Activity}}$ in the range {G('distrib_boost_range_baseline')}. The IL6-gp130-STAT axis is more selective: its boosting effect on (pro)MMP2/9 is in the range {G('IL6_axis_proMMP29_baseline')}, but its effect on proMMP1 is comparatively weak ({G('IL6_axis_proMMP1_baseline')}). $\alpha$SMA is the least regulated phenotype at baseline; no upstream node produces $|\Delta\text{{Activity}}|$ above {F('aSMA_max_abs_baseline')}, indicating that contractile output is largely insensitive to upstream perturbations under low mechanical tension. In contrast to these distributed boosters, LATS1/2 exerts strong braking control over the collagen-related nodes (CI, CImRNA) and proliferation, indicating early suppression of matrix-deposition pathways under low mechanical stimulation.

Taken together, baseline tension defines an activation-dominated regime in which most phenotypic outputs are governed by distributed booster activity, with limited but targeted braking control via LATS1/2. This baseline state provides the reference against which we now describe the more structured regulation that emerges at higher tension.

At high tension level, regulatory control becomes more pronounced and structured, but boosters remain the dominant regulators. The number of phenotypes with at least one brake above threshold rises from {G('n_phen_with_brakes_baseline')}/11 at baseline to {G('n_phen_with_brakes_high')}/11 at high tension, and the affected phenotypes are $\alpha$SMA, proMMP1, MMP1, MMP2, MMP9, and CImRNA, which now have both boosters and brakes in their top-10 influencers (Fig.~S2C). The remaining phenotypes (proMMP2, proMMP9, fibronectin, and proliferation) instead exhibit fewer but stronger boosters.

$\alpha$SMA shifts from being weakly regulated at baseline (max $|\Delta\text{{Activity}}| = {F('aSMA_max_abs_baseline')}$) to being strongly regulated at high tension. Its dominant boosters are the mechanotransduction nodes along the integrin--Rho--ROCK--F-actin--SRF axis ($\Delta\text{{Activity}}$ {G('aSMA_mechano_high')}), and its dominant brakes are TGF$\beta$R and Smad3 ($\Delta\text{{Activity}}$ {G('aSMA_tgfb_brake_high')}).

proMMP1 retains the same set of distributed boosters at high tension (TGF$\beta$, ROS, NOX, AP1, mechanical, NF$\kappa$B), but their influence amplifies: $\Delta\text{{Activity}}$ moves from {G('proMMP1_amp_baseline')} at baseline to {G('proMMP1_amp_high')} at high tension. Smad7 also emerges as a new (non-top-10) booster of proMMP1, with $\Delta\text{{Activity}}$ shifting from ${F('proMMP1_smad7_baseline')}$ to ${F('proMMP1_smad7_high')}$. On the brake side, Smad3 strengthens, with $\Delta\text{{Activity}}$ rising from ${F('proMMP1_smad3_baseline')}$ to ${F('proMMP1_smad3_high')}$. MMP1 shows a different shift: at baseline it is controlled mainly by p38 ($\Delta\text{{Activity}} = {F('MMP1_p38_baseline')}$), and at high tension the dominant boosters become TGF$\beta$ family members and the mechanical input ($\Delta\text{{Activity}}$ {G('MMP1_TGFb_high')}); p38 itself reverses role, moving from a booster ($\Delta\text{{Activity}} = {F('MMP1_p38_baseline')}$) to a modest brake ($\Delta\text{{Activity}} = {F('MMP1_p38_high')}$).

MMP2 remains booster-controlled at high tension, but the responsible nodes change. The IL6-gp130-STAT axis together with NF$\kappa$B and p38 lose influence (booster $\Delta\text{{Activity}}$ shifts from {G('MMP2_baseline_boost')} at baseline to {G('MMP2_high_boost_old')} at high tension), while ROS, NOX, and TGF$\beta$ gain influence ($\Delta\text{{Activity}}$ {G('MMP2_high_boost_new')}). TIMP1 also strengthens as a brake, rising from $\Delta\text{{Activity}} = {F('MMP2_TIMP1_baseline')}$ at baseline to ${F('MMP2_TIMP1_high')}$ at high tension. MMP9 follows a similar redistribution: its dominant booster shifts from upstream IL6-associated signaling at baseline ($\Delta\text{{Activity}}$ {G('MMP9_il6_baseline')}) to NF$\kappa$B at high tension ($\Delta\text{{Activity}} = {F('MMP9_NFKB_high')}$, vs. ${F('MMP9_NFKB_baseline')}$ at baseline), and TIMP1/2 strengthen as brakes (from {G('MMP9_TIMPs_baseline')} to {G('MMP9_TIMPs_high')}).

Collagen transcription (CImRNA) shows a clean redistribution. At baseline its dominant brake is LATS1/2 ($\Delta\text{{Activity}} = {F('CImRNA_LATS_baseline')}$); at high tension this brake weakens substantially ($\Delta\text{{Activity}} = {F('CImRNA_LATS_high')}$). Instead, the mechanotransduction axis (integrin, Rho, ROCK) becomes the main booster ($\Delta\text{{Activity}}$ {G('CImRNA_mechano_high')}), and YAP--TEAD takes over as a modest brake ($\Delta\text{{Activity}}$ {G('CImRNA_YAP_TEAD_high')}). At the protein level, collagen (CI) becomes brake-dominated at high tension. Its main brakes are MMP1, MMP2, and MMP14 ($\Delta\text{{Activity}}$ {G('CI_MMPs_high')}), with additional braking input from TGF$\beta$R, ROS, AP1, and NOX ($\Delta\text{{Activity}}$ {G('CI_upstream_high')}). Booster effects on CI remain weak (CImRNA, integrin--Rho--ROCK, TIMP1: $\Delta\text{{Activity}}$ {G('CI_weak_boost_high')}). As with CImRNA, LATS1/2 weakens from ${F('CI_LATS_baseline')}$ to ${F('CI_LATS_high')}$, but unlike CImRNA, CI is not strongly influenced by YAP/TEAD at high tension ($\Delta\text{{Activity}}$ {G('CI_YAP_TEAD_high')}).

Fibronectin and proliferation move in the opposite direction: their regulator set is pruned rather than expanded. Fibronectin had {G('fibro_n_boost_baseline')} boosters above $|\Delta\text{{Activity}}| \geq 0.25$ at baseline; at high tension this drops to {G('fibro_n_boost_high')}, and the surviving boosters are the mechanical input ($\Delta\text{{Activity}} = {F('fibro_mech_high')}$) and NF$\kappa$B ($\Delta\text{{Activity}} = {F('fibro_NFKB_high')}$). Proliferation is pruned even more sharply, from {G('prol_n_boost_baseline')} boosters above threshold at baseline to {G('prol_n_boost_high')} at high tension, with the mechanical input emerging as the single dominant booster ($\Delta\text{{Activity}} = {F('prol_mech_high')}$).

These shifts are accompanied by the emergence of two regulators that were nearly silent at baseline. Integrin (B1int) appears in {G('B1int_top10_baseline')}/11 phenotype top-10 lists at baseline but in {G('B1int_top10_high')}/11 at high tension, where it acts as a booster for $\alpha$SMA ($\Delta\text{{Activity}} = {F('B1int_aSMA_high')}$) and as a brake for proMMP1 ($\Delta\text{{Activity}} = {F('B1int_proMMP1_high')}$). The mechanical input itself shows the same pattern: from {G('mech_top10_baseline')}/11 phenotype top-10 lists at baseline to {G('mech_top10_high')}/11 at high tension. Together, these changes indicate that increasing tension does not merely amplify existing regulation: it produces three distinct effects --- brakes appear in {int(G('n_phen_with_brakes_high'))-int(G('n_phen_with_brakes_baseline'))} more phenotype top-10 lists than at baseline, several boosters change identity (most notably p38, IL6--gp130--STAT, and LATS1/2), and integrin and the mechanical input emerge as top regulators where they were largely absent before.

At maximum tension, the regulatory architecture is essentially saturated. The mechanical input remains a top-10 influencer in {G('mech_top10_max')}/11 phenotypes (vs.\ {G('mech_top10_high')}/11 at high tension), and the number of phenotypes carrying any brake remains stable at {G('n_phen_with_brakes_max')}/11 (vs.\ {G('n_phen_with_brakes_high')}/11 at high tension). The mean increase in $|\Delta\text{{Activity}}|$ from high to maximum tension across all eleven phenotypes is small, consistent with quantitative amplification rather than the emergence of new regulators (Fig.~\ref{{case3}}D). Collagen outputs remain primarily constrained by inhibitory mechanisms, while cytoskeletal and contractile phenotypes are driven by the same mechanosensitive activation pathways established at high tension.
"""

with open('CM_tensiontext_filled.tex','w') as fh:
    fh.write(TEXT)
print('saved CM_tensiontext_filled.tex')
print()
print(TEXT)

saved CM_tensiontext_filled.tex


Fibroblast signaling and ECM remodeling pathways have been mapped extensively, yet the relative influence of individual regulators within these pathways remains poorly understood, especially under varying mechanical tension. To address this gap, we quantified how each upstream species and input perturbation shapes phenotypic outputs across four tension levels, defined as low ($y_{\text{mechanical}} = 0.01$), baseline ($y_{\text{mechanical}} = b$), high ($y_{\text{mechanical}} = b + p$), and maximum ($y_{\text{mechanical}} = 1$). For each tension level we identify ``brakes'' (nodes whose knockout increases phenotypic activity, $\Delta X_{iKO} > 0$) and ``boosters'' (nodes whose knockout decreases phenotypic activity, $\Delta X_{iKO} < 0$), and quantify their effect on eleven phenotypic nodes ($\alpha$SMA, proMMP1, proMMP2, proMMP9, MMP1, MMP2, MMP9, CImRNA, CI, fibronectin, and proliferation; Fig.~\ref{case3}). On each panel of Fig.~\ref{case3}A--D, the